# Tritium Inventory Parameter Analysis

This notebook analyzes the results from the parallelized tritium inventory simulations using parallel coordinates plots to identify optimal parameter combinations.

## Features:
- Load Parquet data from parameter sweeps
- Create interactive parallel coordinates plots
- Analyze relationships between input parameters and economic losses
- Identify successful startup combinations

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up plotting parameters
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📊 Libraries loaded successfully!")

📊 Libraries loaded successfully!


In [2]:
# Function to load and explore Parquet data
def load_tritium_data(file_pattern="*_points.parquet"):
    """
    Load tritium inventory simulation results from Parquet files
    """
    # Find all Parquet files matching the pattern
    parquet_files = list(Path('.').glob(file_pattern))
    
    if not parquet_files:
        print("❌ No Parquet files found! Run the simulation first.")
        return None
    
    print(f"📁 Found {len(parquet_files)} Parquet file(s):")
    for file in parquet_files:
        print(f"   • {file.name}")
    
    # Load the most recent or largest file
    latest_file = max(parquet_files, key=lambda f: f.stat().st_mtime)
    print(f"\n📈 Loading data from: {latest_file.name}")
    
    df = pd.read_parquet(latest_file)
    
    print(f"\n📊 Dataset Overview:")
    print(f"   • Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   • Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    return df, latest_file.name

# Load the data
df, filename = load_tritium_data()

if df is not None:
    print(f"\n🎯 Column names:")
    for i, col in enumerate(df.columns):
        print(f"   {i+1:2d}. {col}")

📁 Found 1 Parquet file(s):
   • parametric_analysis_results_1x1_points.parquet

📈 Loading data from: parametric_analysis_results_1x1_points.parquet

📊 Dataset Overview:
   • Shape: 1 rows × 24 columns
   • Memory usage: 0.0 MB

🎯 Column names:
    1. V_plasma
    2. T_i
    3. n_tot
    4. tau_p_T
    5. tau_p_He3
    6. P_aux
    7. P_lost_rad
    8. P_aux_all_DT
    9. P_lost_rad_all_DT
   10. TBR_DT
   11. TBR_DDn
   12. tau_ifc
   13. tau_ofc
   14. eta_th
   15. plant_avail
   16. Cost_per_kWh
   17. t_startup
   18. P_fusion_startup
   19. E_lost
   20. Dollar_Lost
   21. P_fusion_DD_startup
   22. P_fusion_DT_startup
   23. avg_n_T_startup
   24. avg_n_D_startup


In [3]:
# Data preprocessing and classification
def classify_simulations(df):
    """
    Classify simulation results into successful/failed categories
    """
    if df is None:
        return None
    
    # Create a copy for analysis
    df_analysis = df.copy()
    
    # Convert startup time from seconds to years for better readability
    df_analysis['t_startup_years'] = df_analysis['t_startup'] / (365.25 * 24 * 3600)
    
    # Convert power to MW
    df_analysis['P_fusion_startup_MW'] = df_analysis['P_fusion_startup'] / 1e6
    
    # Convert dollar losses to millions
    df_analysis['Dollar_Lost_M'] = df_analysis['Dollar_Lost'] / 1e6
    
    # Classify simulations
    successful_mask = (df_analysis['t_startup_years'] < 5) & (df_analysis['t_startup_years'] > 0)
    df_analysis['Status'] = 'Failed'
    df_analysis.loc[successful_mask, 'Status'] = 'Successful'
    
    # Add a color column for plotting
    df_analysis['Color'] = df_analysis['Status'].map({
        'Successful': 'green',
        'Failed': 'red'
    })
    
    print(f"\n📈 Simulation Results Summary:")
    print(f"   • Total simulations: {len(df_analysis):,}")
    print(f"   • Successful (< 5 years): {successful_mask.sum():,} ({100*successful_mask.mean():.1f}%)")
    print(f"   • Failed (≥ 5 years or infinite): {(~successful_mask).sum():,} ({100*(~successful_mask).mean():.1f}%)")
    
    if successful_mask.any():
        successful_df = df_analysis[successful_mask]
        print(f"\n🎯 Successful Simulations Stats:")
        print(f"   • Startup time range: {successful_df['t_startup_years'].min():.2f} - {successful_df['t_startup_years'].max():.2f} years")
        print(f"   • Average startup time: {successful_df['t_startup_years'].mean():.2f} years")
        print(f"   • Dollar losses range: ${successful_df['Dollar_Lost_M'].min():.1f}M - ${successful_df['Dollar_Lost_M'].max():.1f}M")
        print(f"   • Average dollar losses: ${successful_df['Dollar_Lost_M'].mean():.1f}M")
    
    return df_analysis

# Process the data
df_analysis = classify_simulations(df)


📈 Simulation Results Summary:
   • Total simulations: 1
   • Successful (< 5 years): 1 (100.0%)
   • Failed (≥ 5 years or infinite): 0 (0.0%)

🎯 Successful Simulations Stats:
   • Startup time range: 0.20 - 0.20 years
   • Average startup time: 0.20 years
   • Dollar losses range: $0.0M - $0.0M
   • Average dollar losses: $0.0M


In [4]:
# Define input parameters for parallel coordinates plot
def get_input_parameters():
    """
    Define the input parameters for the parallel coordinates plot
    """
    input_params = [
        'V_plasma',         # Plasma volume (m³)
        'T_i',             # Ion temperature (keV)
        'n_tot',           # Total density (m⁻³)
        'tau_p_T',         # Tritium particle confinement time (s)
        'tau_p_He3',       # Helium-3 particle confinement time (s)
        'P_aux',           # Auxiliary power (W)
        'P_lost_rad',      # Radiated power loss (W)
        'TBR_DT',          # DT tritium breeding ratio
        'TBR_DDn',         # DD neutron tritium breeding ratio
        'tau_ifc',         # In-vessel fuel cycle time (s)
        'tau_ofc',         # Out-of-vessel fuel cycle time (s)
        'eta_th',          # Thermal efficiency
        'plant_avail',     # Plant availability
        'Cost_per_kWh'     # Cost per kWh ($/kWh)
    ]
    
    return input_params

# Create normalized version for better visualization
def normalize_for_plotting(df, params):
    """
    Normalize parameters to 0-1 scale for better parallel coordinates visualization
    """
    df_norm = df.copy()
    
    for param in params:
        if param in df_norm.columns:
            param_min = df_norm[param].min()
            param_max = df_norm[param].max()
            if param_max > param_min:  # Avoid division by zero
                df_norm[f'{param}_norm'] = (df_norm[param] - param_min) / (param_max - param_min)
            else:
                df_norm[f'{param}_norm'] = 0.5  # Constant value case
    
    return df_norm

input_parameters = get_input_parameters()
print(f"\n📊 Input parameters for analysis ({len(input_parameters)}):")
for i, param in enumerate(input_parameters):
    print(f"   {i+1:2d}. {param}")


📊 Input parameters for analysis (14):
    1. V_plasma
    2. T_i
    3. n_tot
    4. tau_p_T
    5. tau_p_He3
    6. P_aux
    7. P_lost_rad
    8. TBR_DT
    9. TBR_DDn
   10. tau_ifc
   11. tau_ofc
   12. eta_th
   13. plant_avail
   14. Cost_per_kWh


In [5]:
# Create interactive parallel coordinates plot
def create_parallel_coordinates_plot(df, input_params, target_column='Dollar_Lost_M', max_samples=5000):
    """
    Create an interactive parallel coordinates plot using Plotly
    """
    if df is None:
        print("❌ No data available for plotting")
        return None
    
    # Sample data if too large for interactive plotting
    if len(df) > max_samples:
        print(f"📊 Sampling {max_samples:,} points from {len(df):,} total for interactive plotting")
        
        # Stratified sampling: keep all successful ones, sample failed ones
        successful_df = df[df['Status'] == 'Successful']
        failed_df = df[df['Status'] == 'Failed']
        
        if len(successful_df) > 0:
            print(f"   • Keeping all {len(successful_df):,} successful simulations")
            remaining_samples = max_samples - len(successful_df)
            if remaining_samples > 0 and len(failed_df) > 0:
                failed_sample = failed_df.sample(n=min(remaining_samples, len(failed_df)), random_state=42)
                df_plot = pd.concat([successful_df, failed_sample])
            else:
                df_plot = successful_df
        else:
            df_plot = df.sample(n=min(max_samples, len(df)), random_state=42)
    else:
        df_plot = df.copy()
    
    print(f"\n📈 Creating parallel coordinates plot with {len(df_plot):,} points")
    
    # Prepare dimensions for the plot
    dimensions = []
    
    # Add input parameters
    for param in input_params:
        if param in df_plot.columns:
            dimensions.append(dict(
                label=param,
                values=df_plot[param],
                range=[df_plot[param].min(), df_plot[param].max()]
            ))
    
    # Add the target column (Dollar_Lost_M) as the last axis
    if target_column in df_plot.columns:
        dimensions.append(dict(
            label=f'{target_column} (Target)',
            values=df_plot[target_column],
            range=[df_plot[target_column].min(), df_plot[target_column].max()]
        ))
    
    # Create the parallel coordinates plot
    fig = go.Figure(data=
        go.Parcoords(
            line=dict(
                color=df_plot[target_column],
                colorscale='RdYlGn_r',  # Red (bad) to Green (good)
                showscale=True,
                colorbar=dict(title=f"{target_column}<br>(Million $)")
            ),
            dimensions=dimensions
        )
    )
    
    fig.update_layout(
        title=f"Tritium Inventory Parameter Analysis - {filename}<br>"
              f"<sub>Input Parameters vs {target_column} | "
              f"{len(df_plot):,} simulations | "
              f"{(df_plot['Status'] == 'Successful').sum():,} successful</sub>",
        font_size=12,
        height=600,
        width=1200
    )
    
    return fig

# Create the plot
if df_analysis is not None:
    fig = create_parallel_coordinates_plot(df_analysis, input_parameters, 'Dollar_Lost_M')
    
    if fig is not None:
        fig.show()
        print("\n🎨 Parallel coordinates plot created!")
        print("💡 Usage tips:")
        print("   • Click and drag on any axis to filter data")
        print("   • Green lines = low dollar losses (good)")
        print("   • Red lines = high dollar losses (bad)")
        print("   • Look for parameter ranges that produce green lines")
else:
    print("❌ No data available for plotting")


📈 Creating parallel coordinates plot with 1 points



🎨 Parallel coordinates plot created!
💡 Usage tips:
   • Click and drag on any axis to filter data
   • Green lines = low dollar losses (good)
   • Red lines = high dollar losses (bad)
   • Look for parameter ranges that produce green lines


In [6]:
# Create additional analysis plots
def create_success_analysis_plots(df):
    """
    Create additional plots to analyze successful vs failed simulations
    """
    if df is None:
        return
    
    successful_df = df[df['Status'] == 'Successful']
    
    if len(successful_df) == 0:
        print("❌ No successful simulations found for detailed analysis")
        return
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Startup Time Distribution',
            'Dollar Losses vs Startup Time',
            'Fusion Power vs Ion Temperature',
            'Success Rate by Parameter Ranges'
        ),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # 1. Startup time histogram
    fig.add_trace(
        go.Histogram(
            x=successful_df['t_startup_years'],
            nbinsx=30,
            name='Successful Startups',
            marker_color='green',
            opacity=0.7
        ),
        row=1, col=1
    )
    
    # 2. Dollar losses vs startup time scatter
    fig.add_trace(
        go.Scatter(
            x=successful_df['t_startup_years'],
            y=successful_df['Dollar_Lost_M'],
            mode='markers',
            name='Successful Sims',
            marker=dict(color='green', size=4, opacity=0.6)
        ),
        row=1, col=2
    )
    
    # 3. Fusion power vs ion temperature
    fig.add_trace(
        go.Scatter(
            x=successful_df['T_i'],
            y=successful_df['P_fusion_startup_MW'],
            mode='markers',
            name='Fusion Power',
            marker=dict(
                color=successful_df['Dollar_Lost_M'],
                colorscale='RdYlGn_r',
                size=6,
                showscale=False
            )
        ),
        row=2, col=1
    )
    
    # 4. Success rate analysis (simplified bar chart)
    # Group by T_i ranges and calculate success rates
    df['T_i_range'] = pd.cut(df['T_i'], bins=5)
    success_rate = df.groupby('T_i_range')['Status'].apply(lambda x: (x == 'Successful').mean())
    
    fig.add_trace(
        go.Bar(
            x=[f"{interval.left:.1f}-{interval.right:.1f}" for interval in success_rate.index],
            y=success_rate.values * 100,
            name='Success Rate (%)',
            marker_color='blue',
            opacity=0.7
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title=f"Detailed Analysis of Tritium Inventory Results<br>"
              f"<sub>{len(successful_df):,} successful simulations from {len(df):,} total</sub>",
        height=800,
        showlegend=False
    )
    
    # Update axis labels
    fig.update_xaxes(title_text="Startup Time (years)", row=1, col=1)
    fig.update_xaxes(title_text="Startup Time (years)", row=1, col=2)
    fig.update_xaxes(title_text="Ion Temperature (keV)", row=2, col=1)
    fig.update_xaxes(title_text="T_i Range (keV)", row=2, col=2)
    
    fig.update_yaxes(title_text="Count", row=1, col=1)
    fig.update_yaxes(title_text="Dollar Losses (M$)", row=1, col=2)
    fig.update_yaxes(title_text="Fusion Power (MW)", row=2, col=1)
    fig.update_yaxes(title_text="Success Rate (%)", row=2, col=2)
    
    return fig

# Create the analysis plots
if df_analysis is not None:
    analysis_fig = create_success_analysis_plots(df_analysis)
    if analysis_fig is not None:
        analysis_fig.show()
        print("\n📊 Detailed analysis plots created!")


📊 Detailed analysis plots created!


In [7]:
# Export optimal parameter combinations
def export_optimal_combinations(df, top_n=10):
    """
    Export the best parameter combinations for further analysis
    """
    if df is None:
        return
    
    successful_df = df[df['Status'] == 'Successful']
    
    if len(successful_df) == 0:
        print("❌ No successful combinations to export")
        return
    
    # Sort by startup time (ascending) and dollar losses (ascending)
    optimal_df = successful_df.sort_values(['t_startup_years', 'Dollar_Lost_M']).head(top_n)
    
    print(f"\n🏆 Top {len(optimal_df)} Optimal Parameter Combinations:")
    print("=" * 80)
    
    # Display key metrics
    display_columns = [
        'T_i', 'n_tot', 'P_aux', 'TBR_DT', 'tau_p_T', 
        't_startup_years', 'P_fusion_startup_MW', 'Dollar_Lost_M'
    ]
    
    for i, (idx, row) in enumerate(optimal_df.iterrows()):
        print(f"\n🎯 Combination {i+1}:")
        print(f"   • Ion Temperature: {row['T_i']:.1f} keV")
        print(f"   • Density: {row['n_tot']:.1e} m⁻³")
        print(f"   • Aux Power: {row['P_aux']/1e6:.1f} MW")
        print(f"   • TBR_DT: {row['TBR_DT']:.3f}")
        print(f"   • Startup Time: {row['t_startup_years']:.2f} years")
        print(f"   • Fusion Power: {row['P_fusion_startup_MW']:.1f} MW")
        print(f"   • Dollar Losses: ${row['Dollar_Lost_M']:.1f}M")
    
    # Save to CSV for further analysis
    output_file = f"optimal_combinations_{filename.replace('.parquet', '.csv')}"
    optimal_df.to_csv(output_file, index=False)
    print(f"\n💾 Optimal combinations saved to: {output_file}")
    
    return optimal_df

# Export optimal combinations
if df_analysis is not None:
    optimal_combinations = export_optimal_combinations(df_analysis, top_n=10)


🏆 Top 1 Optimal Parameter Combinations:

🎯 Combination 1:
   • Ion Temperature: 17.0 keV
   • Density: 1.7e+20 m⁻³
   • Aux Power: 60.0 MW
   • TBR_DT: 1.100
   • Startup Time: 0.20 years
   • Fusion Power: 78.1 MW
   • Dollar Losses: $0.0M

💾 Optimal combinations saved to: optimal_combinations_parametric_analysis_results_1x1_points.csv


In [8]:
# Summary and recommendations
def generate_summary_report(df):
    """
    Generate a comprehensive summary report
    """
    if df is None:
        return
    
    print("\n" + "=" * 80)
    print("🏁 TRITIUM INVENTORY ANALYSIS SUMMARY REPORT")
    print("=" * 80)
    
    successful_df = df[df['Status'] == 'Successful']
    
    print(f"\n📊 Dataset Overview:")
    print(f"   • Total parameter combinations tested: {len(df):,}")
    print(f"   • Successful combinations (< 5 years): {len(successful_df):,}")
    print(f"   • Success rate: {len(successful_df)/len(df)*100:.2f}%")
    print(f"   • Data file: {filename}")
    
    if len(successful_df) > 0:
        print(f"\n🎯 Successful Simulations Analysis:")
        print(f"   • Fastest startup: {successful_df['t_startup_years'].min():.2f} years")
        print(f"   • Slowest startup: {successful_df['t_startup_years'].max():.2f} years")
        print(f"   • Average startup: {successful_df['t_startup_years'].mean():.2f} years")
        print(f"   • Minimum dollar losses: ${successful_df['Dollar_Lost_M'].min():.1f}M")
        print(f"   • Maximum dollar losses: ${successful_df['Dollar_Lost_M'].max():.1f}M")
        print(f"   • Average dollar losses: ${successful_df['Dollar_Lost_M'].mean():.1f}M")
        
        # Parameter ranges for successful combinations
        print(f"\n🔧 Optimal Parameter Ranges (from successful combinations):")
        key_params = ['T_i', 'n_tot', 'P_aux', 'TBR_DT', 'tau_p_T']
        for param in key_params:
            if param in successful_df.columns:
                min_val = successful_df[param].min()
                max_val = successful_df[param].max()
                print(f"   • {param}: {min_val:.2e} - {max_val:.2e}")
        
        # Best combination
        best_combo = successful_df.loc[successful_df['t_startup_years'].idxmin()]
        print(f"\n🏆 Best Combination (fastest startup):")
        print(f"   • Startup time: {best_combo['t_startup_years']:.2f} years")
        print(f"   • Dollar losses: ${best_combo['Dollar_Lost_M']:.1f}M")
        print(f"   • Ion temperature: {best_combo['T_i']:.1f} keV")
        print(f"   • Density: {best_combo['n_tot']:.1e} m⁻³")
        print(f"   • TBR_DT: {best_combo['TBR_DT']:.3f}")
    
    else:
        print(f"\n❌ No successful combinations found in this parameter space!")
        print(f"\n💡 Recommendations:")
        print(f"   • Try expanding parameter ranges")
        print(f"   • Consider different operational scenarios")
        print(f"   • Increase auxiliary power ranges")
        print(f"   • Optimize tritium breeding ratios")
    
    print("\n" + "=" * 80)
    print("📈 Analysis complete! Use the parallel coordinates plot to explore parameter relationships.")
    print("=" * 80)

# Generate summary report
if df_analysis is not None:
    generate_summary_report(df_analysis)


🏁 TRITIUM INVENTORY ANALYSIS SUMMARY REPORT

📊 Dataset Overview:
   • Total parameter combinations tested: 1
   • Successful combinations (< 5 years): 1
   • Success rate: 100.00%
   • Data file: parametric_analysis_results_1x1_points.parquet

🎯 Successful Simulations Analysis:
   • Fastest startup: 0.20 years
   • Slowest startup: 0.20 years
   • Average startup: 0.20 years
   • Minimum dollar losses: $0.0M
   • Maximum dollar losses: $0.0M
   • Average dollar losses: $0.0M

🔧 Optimal Parameter Ranges (from successful combinations):
   • T_i: 1.70e+01 - 1.70e+01
   • n_tot: 1.70e+20 - 1.70e+20
   • P_aux: 6.00e+07 - 6.00e+07
   • TBR_DT: 1.10e+00 - 1.10e+00
   • tau_p_T: 2.55e+00 - 2.55e+00

🏆 Best Combination (fastest startup):
   • Startup time: 0.20 years
   • Dollar losses: $0.0M
   • Ion temperature: 17.0 keV
   • Density: 1.7e+20 m⁻³
   • TBR_DT: 1.100

📈 Analysis complete! Use the parallel coordinates plot to explore parameter relationships.
